<a href="https://colab.research.google.com/github/testimonyadewale/msc-grocery-forecasting/blob/main/MSc_models_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully')

# Load the processed dataset — no need to redo feature engineering
df = pd.read_csv('/content/drive/MyDrive/MSc_Data/train_processed.csv')
df['date'] = pd.to_datetime(df['date'])

print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All libraries loaded successfully
Loaded: (730500, 20)
Columns: ['date', 'store', 'item', 'sales', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_365', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_84', 'rolling_std_7', 'rolling_std_28', 'day_of_week', 'week_of_year', 'month', 'year', 'quarter', 'is_weekend']


In [20]:
feature_cols = [
    'store', 'item',
    'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_365',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_84',
    'rolling_std_7', 'rolling_std_28',
    'day_of_week', 'week_of_year', 'month',
    'year', 'quarter', 'is_weekend'
]

X = df[feature_cols]
y = df['sales']

train_mask = df['date'] < '2017-01-01'
test_mask  = df['date'] >= '2017-01-01'

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Training rows: {len(X_train)}')
print(f'Test rows:     {len(X_test)}')

Training rows: 548000
Test rows:     182500


In [21]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{name}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')
    return {'Model': name, 'MAE': round(mae,4),
            'RMSE': round(rmse,4), 'R2': round(r2,4)}

print('evaluate function ready')

evaluate function ready


In [22]:
ma_pred = X_test['rolling_mean_28'].fillna(y_train.mean())
results_ma = evaluate('Moving Average (Baseline)', y_test, ma_pred)

Moving Average (Baseline): MAE=9.2373  RMSE=12.2565  R2=0.8491


In [23]:
# Run ARIMA on one store-item series
# Filter to store 1, item 1 for simplicity
series = df[(df['store']==1) & (df['item']==1)].sort_values('date')

arima_train = series[series['date'] < '2017-01-01']['sales']
arima_test  = series[series['date'] >= '2017-01-01']['sales']

# Fit ARIMA with order (1,1,1) - a standard starting point
arima_model = ARIMA(arima_train, order=(1,1,1))
arima_fit   = arima_model.fit()

# Forecast the same number of steps as the test set
arima_pred = arima_fit.forecast(steps=len(arima_test))

results_arima = evaluate_model('ARIMA', arima_test, arima_pred)

ARIMA:
  MAE:  6.7801
  RMSE: 8.5935
  R2:   -0.5316


In [24]:
# Create and train the Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

#Make predictions on the test set
rf_pred     = rf.predict(X_test)
results_rf  = evaluate('Random Forest', y_test, rf_pred)


Random Forest: MAE=6.3303  RMSE=8.2420  R2=0.9318


In [25]:
# Create and train the XGBoost model
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
xgb_pred = xgb_model.predict(X_test)

results_xgb = evaluate_model('XGBoost', y_test, xgb_pred)


XGBoost:
  MAE:  6.1741
  RMSE: 8.0290
  R2:   0.9352


In [26]:
# Combine all results into one comparison table
results_df = pd.DataFrame([results_ma, results_arima, results_rf, results_xgb])

# Add improvement vs baseline column
baseline_mae = results_ma['MAE']
results_df['MAE_improvement_%'] = (
    (baseline_mae - results_df['MAE']) / baseline_mae * 100
).round(1)

print('\n===== MODEL COMPARISON TABLE =====')
print(results_df.to_string(index=False))

# Save to CSV for Power BI
results_df.to_csv('/content/drive/MyDrive/MSc_Data/model_results.csv', index=False)
print('\nResults saved to model_results.csv')



===== MODEL COMPARISON TABLE =====
                    Model    MAE    RMSE      R2  MAE_improvement_%
Moving Average (Baseline) 9.2373 12.2565  0.8491                0.0
                    ARIMA 6.7801  8.5935 -0.5316               26.6
            Random Forest 6.3303  8.2420  0.9318               31.5
                  XGBoost 6.1741  8.0290  0.9352               33.2

Results saved to model_results.csv


In [27]:
# Plot actual vs predicted for one store and item
sample = df[(df['store']==1) & (df['item']==1) & (df['date'] >= '2017-01-01')]
sample = sample.sort_values('date')

# Get XGBoost predictions for this sample
sample_features = sample[feature_cols]
sample_pred_xgb = xgb_model.predict(sample_features)
sample_pred_ma  = sample['rolling_mean_28'].fillna(y_train.mean())

fig = go.Figure()
fig.add_trace(go.Scatter(x=sample['date'], y=sample['sales'],
              name='Actual Sales', line=dict(color='black')))
fig.add_trace(go.Scatter(x=sample['date'], y=sample_pred_xgb,
              name='XGBoost Forecast', line=dict(color='blue', dash='dash')))
fig.add_trace(go.Scatter(x=sample['date'], y=sample_pred_ma,
              name='Moving Average Baseline', line=dict(color='red', dash='dot')))

fig.update_layout(title='Actual vs Predicted — Store 1, Item 1 (2017)',
                  xaxis_title='Date', yaxis_title='Units Sold')
fig.show()


In [28]:
# ── INVENTORY SIMULATION ──
# I compare two scenarios:
# Scenario A: retailer uses Moving Average to decide order quantity
# Scenario B: retailer uses XGBoost (or best model) to decide order quantity

# Get the test set with actual sales and both predictions
sim_df = df[test_mask].copy()
sim_df['pred_ma']  = ma_pred.values
sim_df['pred_xgb'] = xgb_pred

# ── WASTE CALCULATION ──
# Waste = units ordered minus units sold (when order > actual)
# This represents unsold stock that would be discarded
sim_df['waste_ma']  = (sim_df['pred_ma']  - sim_df['sales']).clip(lower=0)
sim_df['waste_xgb'] = (sim_df['pred_xgb'] - sim_df['sales']).clip(lower=0)

# ── STOCKOUT CALCULATION ──
# Stockout = units of demand not met (when actual > order)
sim_df['stockout_ma']  = (sim_df['sales'] - sim_df['pred_ma'] ).clip(lower=0)
sim_df['stockout_xgb'] = (sim_df['sales'] - sim_df['pred_xgb']).clip(lower=0)

# ── RESULTS ──
total_waste_ma   = sim_df['waste_ma'].sum()
total_waste_xgb  = sim_df['waste_xgb'].sum()
total_stock_ma   = sim_df['stockout_ma'].sum()
total_stock_xgb  = sim_df['stockout_xgb'].sum()

waste_reduction    = (total_waste_ma  - total_waste_xgb)  / total_waste_ma  * 100
stockout_reduction = (total_stock_ma  - total_stock_xgb)  / total_stock_ma  * 100

print('===== INVENTORY SIMULATION RESULTS =====')
print(f'Total waste   — Moving Average: {total_waste_ma:.0f} units')
print(f'Total waste   — XGBoost:        {total_waste_xgb:.0f} units')
print(f'Waste reduction:                {waste_reduction:.1f}%')
print(f'')
print(f'Total stockout — Moving Average: {total_stock_ma:.0f} units')
print(f'Total stockout — XGBoost:        {total_stock_xgb:.0f} units')
print(f'Stockout reduction:              {stockout_reduction:.1f}%')

# Save simulation results for Power BI
sim_summary = pd.DataFrame({
    'Metric':    ['Waste (units)', 'Stockout (units)'],
    'Moving Average': [total_waste_ma, total_stock_ma],
    'XGBoost':   [total_waste_xgb, total_stock_xgb]
})
sim_summary.to_csv('/content/drive/MyDrive/MSc_Data/simulation_results.csv', index=False)
print('\nSimulation results saved.')


===== INVENTORY SIMULATION RESULTS =====
Total waste   — Moving Average: 834085 units
Total waste   — XGBoost:        580836 units
Waste reduction:                30.4%

Total stockout — Moving Average: 851730 units
Total stockout — XGBoost:        545935 units
Stockout reduction:              35.9%

Simulation results saved.
